In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# In this estimated salary is going to be the output feature and remaining will be the independent feature
# That's the reason why output feature is a continuous variable and it then turns out to be a regression problem
data = data.drop(['RowNumber', 'CustomerId','Surname'],axis=1)


In [4]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [8]:
# Encode the Geography using One-Hot encoder
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
# Combining onehot encoded columns with the data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [ ]:
# Splitting the data into independent features and target feature
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']



In [12]:
# Splitting the data in train-test set
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [13]:
# Scaling down the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
with open('label_encoder_gender_reg.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo_reg.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

with open('scaler_reg.pkl','wb') as file:
    pickle.dump(scaler,file)

In [18]:
# Train ANN for Regression

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard 
import datetime

In [16]:
# Building the model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), # Input -> Hidden layer 1
    Dense(32,activation='relu'), # Hidden layer 1 -> Hidden layer 2
    Dense(1) # Hidden layer 2 -> Output Default activation function applied here is Linear Activation function
])

d:\GenAICourse\ANN-BinaryClassification\annenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
# Compiling the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Setting up TensorBoard
log_dir = 'regressionlogs/fit/' +datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [20]:
# Setting up early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [21]:
history = model.fit(
    X_train, y_train,
    validation_data = (X_test,y_test),
    epochs=100,
    callbacks=[early_stopping_callback,tensorboard_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 100374.9297 - mae: 100374.9297 - val_loss: 98507.5781 - val_mae: 98507.5781
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 99599.2578 - mae: 99599.2578 - val_loss: 96943.4922 - val_mae: 96943.4922
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 96883.3125 - mae: 96883.3125 - val_loss: 92952.7188 - val_mae: 92952.7188
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 91549.1172 - mae: 91549.1172 - val_loss: 86332.4531 - val_mae: 86332.4531
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 83846.6641 - mae: 83846.6641 - val_loss: 77802.8906 - val_mae: 77802.8906
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 74767.6797 - mae: 74767.6797 - val_loss: 68910.7969 - val_mae: 68910.7969
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 65923.0781 - mae: 65923.0781 - val_loss: 61207.0781 - val_mae: 61207.0781
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step 

In [22]:
%load_ext tensorboard

In [26]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6007 (pid 33024), started 0:00:12 ago. (Use '!kill 33024' to kill it.)

In [27]:
# Evaluate model on test data
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f"Test MAE : {test_mae:.2f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 50270.2969 - mae: 50270.2969
Test MAE : 50270.30


In [29]:
model.save('regression_model.keras')